In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords


nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab") 

[nltk_data] Downloading package stopwords to /home/hammad/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/hammad/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/hammad/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
df = pd.read_csv("data/spam.csv")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df.rename(columns={'v1':'target', 'v2':'text'}, inplace=True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


#Data Preprocessing#

In [5]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])
df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.duplicated().sum()

np.int64(403)

In [7]:
len(df)

5572

In [8]:
df = df.drop_duplicates(keep='first')
len(df)

5169

Feature Engineering

In [9]:
from nltk.stem.porter import PorterStemmer
import string
ps = PorterStemmer()
stop_words = set(stopwords.words("english"))

In [10]:
def transform_text(text):
    text = text.lower()
    text = nltk.word_tokenize(text)
    y = []
    for t in text:
        if t.isalnum():
            y.append(t)
    text = y[:]
    y.clear()

    for i in text:
        #moved wordbag out of the look and this function which reduced the processing time by almost 90%
        if i not in stop_words and i not in string.punctuation:
            y.append(i)

    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))
    return " ".join(y)

In [11]:
transform_text("Go until jurong point, crazy.. Available only in bugis n great world la e buffet.. Cine there got amore wat...")

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [12]:
#using list comprehension which is faster then pandas apply function
df['transformed_text'] = [transform_text(t) for t in df['text']]

In [13]:
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [14]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
tfidf = TfidfVectorizer(max_features=500)

In [15]:
x = tfidf.fit_transform(df['transformed_text'])
y = df['target'].values

In [ ]:

x.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(5169, 500))

#Train Test Split#

In [16]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.20, random_state=2)

#Model Training#

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

In [19]:
clfs = {
    "svc": SVC(kernel="sigmoid", gamma=1.0),
    "knc": KNeighborsClassifier(),
    "mnb": MultinomialNB(),
    "dtc": DecisionTreeClassifier(),
    "lrc": LogisticRegression(solver='liblinear', penalty='l1'),
    "rfc": RandomForestClassifier(n_estimators=50, random_state=2),
    "abc": AdaBoostClassifier(n_estimators=50, random_state=2),
    "bc": BaggingClassifier(n_estimators=50, random_state=2),
    "etc": ExtraTreesClassifier(n_estimators=50, random_state=2),
    "gbdt": GradientBoostingClassifier(n_estimators=50, random_state=2),
    "xgb": XGBClassifier(n_estimators=50, random_state=2)
}

#Model Evaluation#

In [ ]:
from sklearn.metrics import accuracy_score, precision_score
def train_classifier(clfs, x_train, y_train, x_test, y_test):
    clfs.fit(x_train, y_train)
    y_pred = clfs.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    return accuracy, precision

In [26]:
accuracy_scores = []
precision_scores = []
for name, clf in clfs.items():
    acs, ps = train_classifier(clfs=clf, x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test)
    accuracy_scores.append(acs)
    precision_scores.append(ps)
    print(f"Model:{name} | Accuracy:{acs} | Precision:{ps}")

Model:svc | Accuracy:0.9671179883945842 | Precision:0.9333333333333333
Model:knc | Accuracy:0.9274661508704062 | Precision:1.0
Model:mnb | Accuracy:0.9709864603481625 | Precision:0.9655172413793104
Model:dtc | Accuracy:0.9584139264990329 | Precision:0.8625954198473282
Model:lrc | Accuracy:0.9632495164410058 | Precision:0.9629629629629629


/home/hammad/.pyenv/versions/mlops/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/hammad/.pyenv/versions/mlops/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Model:rfc | Accuracy:0.9700193423597679 | Precision:0.9421487603305785
Model:abc | Accuracy:0.9235976789168279 | Precision:0.8734177215189873
Model:bc | Accuracy:0.9622823984526112 | Precision:0.9024390243902439
Model:etc | Accuracy:0.9709864603481625 | Precision:0.921875
Model:gbdt | Accuracy:0.9497098646034816 | Precision:0.93
Model:xgb | Accuracy:0.9700193423597679 | Precision:0.9495798319327731


In [27]:
print(f"Max Precision:{max(precision_scores)}")
print(f"Max Accuracy:{max(accuracy_scores)}")

Max Precision:1.0
Max Accuracy:0.9709864603481625
